# optimizer-state-tensor-buffers — worked example 2: Adam init: two moment buffers and a step counter

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `optimizer-state-tensor-buffers`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

Adam maintains three pieces of state per optimization: a first-moment buffer `m` (running mean of gradients), a second-moment buffer `v` (running mean of squared gradients), and a scalar step counter `t` used for bias correction. Both `m` and `v` are allocated with `t.zeros_like(p)` for each parameter. The step counter starts at 0 and is a plain Python int — it increments by 1 each `.step()` call.

## Worked solution

**Step 1 — Materialize params and allocate `m`.**
`self.params = list(params)` then `self.m = [t.zeros_like(p) for p in self.params]`. Each entry matches the corresponding parameter's shape, dtype, and device.

**Step 2 — Allocate `v` separately.**
`self.v = [t.zeros_like(p) for p in self.params]`. This must be a separate list from `self.m`. A common mistake is `self.v = self.m` — that creates an alias, so updating `m` would also update `v`.

**Step 3 — Initialize the step counter.**
`self.t = 0`. This is a plain Python int. Adam increments it once per step and uses it in the bias-correction terms `(1 - beta1**t)` and `(1 - beta2**t)`.

**Step 4 — Verify no aliasing.**
For every index `i`, `id(self.m[i]) != id(self.v[i])`. We mutate `m[0]` and confirm `v[0]` is unaffected.

In [ ]:
import torch as t
import torch.nn as nn

class AdamInit:
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8):
        self.params = list(params)
        self.lr     = lr
        self.betas  = betas
        self.eps    = eps
        # First moment
        self.m = [t.zeros_like(p) for p in self.params]
        # Second moment — SEPARATE allocation, not an alias
        self.v = [t.zeros_like(p) for p in self.params]
        # Step counter
        self.t = 0

# --- exercise it ---
t.manual_seed(0)
model = nn.Linear(6, 4)
adam = AdamInit(model.parameters())

print(f'len(m)={len(adam.m)}, len(v)={len(adam.v)}, t={adam.t}')
assert len(adam.m) == len(adam.v) == len(adam.params)
assert isinstance(adam.t, int) and adam.t == 0

# Verify no aliasing
for i, (mi, vi) in enumerate(zip(adam.m, adam.v)):
    assert id(mi) != id(vi), f'Buffer {i}: m and v must be separate tensors'

# Mutate m[0], v[0] should be unaffected
adam.m[0].fill_(99.0)
assert adam.v[0].abs().max().item() == 0.0, 'v should not be affected by mutating m'

print('Adam init: no aliasing, step counter correct.')